# 17 · Cell registration (CellReg) — from the MATLAB run to a populated schema

**Purpose.** How longitudinal cell identity gets from a CellReg run on the server into
`cell_registration`, and how to query it afterwards. Ends with a worked example on one of
Leon's gratings sites.

> **Read this before running anything.**
> The `cell_registration` schema is **declared in code but not on the server**. The cells
> below that would *create* tables or *insert* rows are marked
> **`# REQUIRES AUTHORISATION`** and are deliberately left un-run. Creating the schema and
> populating it are production writes and need Tobias's explicit go-ahead in that session.
> Everything in §6 (querying) and §7 (the example) is read-only.

**Design review:** `adamacs/schemas/CELL_REGISTRATION_SCHEMA_DESIGN_REVIEW.md`
**Wiki:** `Database/Schemas/roselab_cell_registration.md`


---
## 1. Why this schema exists

Cross-session identity is currently one column: `imaging.Segmentation.Mask.roi_group`.
It is a `smallint`, it is **not part of the primary key**, and it is overwritten in place by
whichever CellReg ingest ran last. Nothing records which run produced it, with what
parameters, or what it held before.

That is wiki issue **D1** — counts recorded by an analysis (111, 146) disagree with the live
database (116, 120) — and **T1**: the delta cannot be reconstructed. It is also why
`adamacs_leon/helpers/pipeline_blocks.py` has `realign_global_ids_to_current_roi_group()`,
whose job is to repair identity after a re-ingest by *guessing* which CellReg column belongs
to which scan.

**What `roi_group` is, in CellReg terms:** the 0-based **row index of `cell_to_index_map`**.
This schema calls that `cell_index`. `roi_group` is now treated as a *derived projection* of
the latest curated run — a cache whose source is recorded, not an independent fact.

The one thing that makes an additive fix possible: a re-ingest rewrites `roi_group` but
**never the mask numbering**, so `(session_id, scan_id, paramset_idx, curation_id, mask)`
is a stable anchor.


---
## 2. Stage 1 — the MATLAB CellReg run on the server

Run on **Loki or ibehaveGPU, never tatchu3** (Tobias, 2025-08-26; a CellReg run exhausted
tatchu3's RAM in Feb 2025).

`Leon_CellReg.m` reads `converted_*.mat` (suite2p `Fall.mat` → CellReg footprints) and writes
into `<CR folder>/Results/`:

| File | Contents |
|---|---|
| `cellRegistered_<yyyymmdd_HHMMss>.mat` | `cell_registered_struct`: `cell_to_index_map`, the four score vectors, `p_same_registered_pairs`, `is_cell_in_overlapping_FOV`, centroids, alignment translations/rotations |
| `logFile_<yyyymmdd_HHMMss>.txt` | **the parameters and the run-level accuracy** |
| `Figures/` | CellReg's own diagnostics |

**Three facts that shape the importer** (measured over 108 files, 18.8 GB):

1. **The `.mat` contains no parameters at all.** They live only in `logFile_*.txt`, whose
   timestamp *trails* the result file (76 s in a verified pair), so pairing is "nearest log
   at or after the result file in the same directory".
2. **The score fields exist only for `registration_approach = 'Probabilistic'`**, and
   `alignment_rotations` only for `'Translations and Rotations'` (86 of 107 files).
3. **Footprints must stay on disk.** 93 of 107 files embed a stack; the largest is
   `513×513×3363` ≈ **7.1 GB** as float64. The importer never loads that field.


In [ ]:
import datetime
import os
import re

import datajoint as dj
import numpy as np

from adamacs.pipeline import imaging, scan, session, event    # activates the Elements
from adamacs.schemas import cell_registration as creg          # declares NOTHING on import
from adamacs.ingest import cellreg                             # the parsers

print("cell_registration activated on import:", creg.schema.is_activated())
print("parser version:", creg.PARSER_VERSION)

---
## 3. Inspect a run's files before touching the database

All of this is pure file reading — no connection needed.


In [ ]:
CR = ("/datajoint-data/data/leonk/analysis/cellreg/"
      "LE_ROS-1881_sess9FPXNI8U_Gratings_CR/Results/cellRegistered_20260129_075504.mat")

print("MAT format :", cellreg.detect_mat_format(CR))
print("fields     :", cellreg.list_fields(CR))
print("footprints :", cellreg.probe_footprints_storage(CR))   # metadata only; stack never read

log = cellreg.find_log_file(CR)
print("paired log :", log)

meta = cellreg.parse_log_file(open(log).read())
print("sessions   :", meta["n_sessions"], "| approach:", meta.get("registration_approach"))
print("alignment  :", meta.get("alignment_type"))
print("accuracy   : FP", meta.get("frac_false_positives"),
      "FN", meta.get("frac_false_negatives"),
      "uncertain", meta.get("frac_uncertain_pairs"))

# the log lists sessions in COLUMN order and the filenames carry the scan id
for i, f in enumerate(meta["session_files"]):
    print(f"   column {i}: {cellreg.scan_id_from_source_file(f)}")

---
## Run 1 — what was actually ingested, and what could not be

Run 1 was authorised on 2026-10-20 and registered **every** CellReg result file on the
host, in the chronological order the registrations were produced. 114 files on disk
collapse to **102 distinct runs** by sha256 — the duplicates are byte-identical backups
scattered across scan folders, which is why a run is keyed by the *content* of its result
file and the other paths live in `CellRegRun.Copy`. One file is unreadable
(`OSError: could not read bytes`), leaving **101 runs** spanning 2023-07-06 to 2026-10-18.

| Table | Rows |
|---|---:|
| `CellRegRun` | 101 |
| `CellRegRun.Session` (resolved columns) | 1,102 |
| `CellRegRun.UnresolvedColumn` | 164 |
| `CellRegRun.Quality` | 101 |
| `CellRegRun.Copy` | 12 |
| `CellRegParamSet` | 68 |
| `RoiGroupProjection`, `RoiGroupSnapshot` | **0, 0** |

1,102 + 164 = 1,266 — every column is accounted for exactly once, it equals
`SUM(n_sessions)` over the 101 runs, and no column appears in both tables.

### How each column found its curation

| Columns | Outcome |
|---:|---|
| 248 | `log_file` — the scan has exactly one curation, so it is forced |
| 317 | `roi_group_match` — a **unique exact** content match |
| 537 | `mask_set_identical` — candidates matched *and* were byte-identical |
| **1,102** | **resolved** → `CellRegRun.Session` |
| 50 | no curation's mask count matches the file's centroid count |
| 25 | candidates match the count but hold **different** mask sets |
| 65 | the column contains no cells |
| 24 | no scan id recoverable from the log |
| **164** | **unresolved** → `CellRegRun.UnresolvedColumn` |
| **1,266** | total = `SUM(n_sessions)` over the 101 runs |

These are the figures in the live schema **after** the demotion described below. Two
earlier drafts of this notebook were wrong here and both errors are worth knowing.
The first quoted 307 and
533 here, which were the *dry-run* counts taken over 99 runs: two runs whose log session
count contradicts their index map initially refused to parse, and the orientation fallback
that rescued them added 13 columns (2 + 11). Both tables must come from the same run —
they did not, and the breakdown then summed to 1,254 against a stated total of 1,266.
The second quoted 573/693, taken before eight columns were demoted; any count copied out
of a live table goes stale the moment the table is corrected, so the reconciliation cell
below re-derives all of them rather than restating them.

### Eight columns demoted: a single candidate is not evidence

`MaskCoordinates` refuses a column when `centroid_locations_corrected` has a different
number of rows than the curation has masks — *"the input file is not this curation"*. Run
1's registrar did not check, so it wrote eight `Session` rows asserting a provenance the
data refuses, and the populate then failed on them.

Every one of the eight was attached by `log_file`: the **forced** case, where the scan has
exactly one curation and the column was attached with no content evidence at all. Two
shapes of fault:

| | |
|---|---|
| off-by-one | 1034 vs 1035, 1280 vs 1279, 1115 vs 1116 — the curation gained or lost an ROI after the run |
| wholesale | 316 vs 910, 370 vs 1088, 336 vs 1064 — the input file is a different segmentation entirely |

They are now in `UnresolvedColumn` with both counts recorded, and the registrar performs
the same comparison before writing a `Session` row. The lesson generalises: `log_file`
resolution means *only one candidate existed*, which is not the same as evidence that it
is the right one.

**The headline result of run 1 is what could NOT be attached, and why.** They are not a parser failure. `roi_group`
is a `smallint` outside the primary key that each run overwrote in place, so for any
curation only the *last* run to target it survives; every earlier run's cross-session
identities are simply gone. Run 1 is the first time that loss has been counted.

The 69 ties are the other honest limit. When two curations hold byte-identical masks
*and* the same `roi_group`, the content match cannot single one out — so the column stays
unresolved with its candidates attached as evidence, rather than being attached to an
arbitrary one of them.

Both are fixed going forward, not retrospectively: `CellRegTask.Session` makes the
operator declare the curation per column *before* the run, and the exporter writes that
manifest beside the footprints. See notebook 18.

### The tie-break, applied 2026-10-20

`roi_group` not matching says only that a *later* run overwrote it, never that the file is
unusable. The operative question is whether the file's per-column centroid count matches a
candidate curation's mask count — and for **562** of the original 701 unresolved columns,
*several* candidates matched. The shortfall was nameability, not incompatibility.

Of those 562, **537 had candidates that were byte-identical in mask set** — same masks,
same centroids — so naming any of them yields the same data. Those are now attached as
`resolution_method='mask_set_identical'`, picking the latest candidate that predated the
run (a curation created afterwards cannot have been its input; 533 that way, 4 by an
earliest-matching fallback).

The remaining **25 have candidates that genuinely differ** and are deliberately left
unresolved: choosing there would be choosing between different answers.

| | before | after |
|---|---:|---:|
| resolved columns | 565 | 1,102 |
| memberships with a curation key | 717,053 (46.6 %) | 1,457,704 (94.6 %) |
| cells with every session attached | 254,150 (42.6 %) | 553,710 (92.7 %) |

Worth knowing *why* the narrowness matters: earliest-matching and latest-predating-the-run
**disagree on 284 of 558 columns**. A convention here is not cosmetic. It is safe only
because it was restricted to candidates that are provably the same mask set.
`scripts/attach_mask_set_identical.py` is the exact operation, re-runnable with `DRY=1`.

In [ ]:
# Re-derive the census from the live schema. Do not trust the numbers written above:
# eight columns were demoted after they were first published, and any count copied out
# of a live table goes stale the moment the table is corrected.
import collections

n_s = len(cr.CellRegRun.Session())
n_u = len(cr.CellRegRun.UnresolvedColumn())
total = sum(cr.CellRegRun.fetch('n_sessions'))
print(f'runs             {len(cr.CellRegRun()):7d}')
print(f'resolved columns {n_s:7d}')
print(f'unresolved       {n_u:7d}')
print(f'total            {n_s + n_u:7d}   SUM(n_sessions) = {total}   reconciles: {n_s + n_u == total}')
print(f'paramsets        {len(cr.CellRegParamSet()):7d}')
print(f'roi_group proj.  {len(cr.RoiGroupProjection()):7d}   <- stays empty by design')
print()
print('resolution_method:',
      dict(collections.Counter(cr.CellRegRun.Session.fetch('resolution_method'))))
print()
for reason, n in collections.Counter(
        r.split(' (')[0][:58] for r in
        cr.CellRegRun.UnresolvedColumn.fetch('reason')).most_common(8):
    print(f'{n:5d}  {reason}')

### Pair-quality completeness

`CellRegResult.PairQuality` stores one row per `(cell, column_a, column_b)` where the cell
is present in **both** columns — that restriction is what keeps the table at ~4.8 M rows
lab-wide instead of cells × all-pairs (88 M).

It is legitimately **shorter** than the index map implies, because CellReg emits `NaN`
`p_same` for some pairs it nonetheless calls the same cell. `n_pairs_expected` and
`n_pairs_nonfinite` record that difference so it can never be mistaken for a parser fault.
On the 11-session ROS-1881 run: **38,921 implied = 38,767 stored + 154 NaN**.

That run is also the acceptance test for the whole parse path, because it was reproduced
independently *before* this code existed. All eleven checks are exact: 4,107 cells, 13,532
memberships, 13,532 mask coordinates, 38,767 pair-quality rows, 306 cells present in all
11 sessions, `min_p_same` populated on all 4,107.

Getting there cost one real bug, worth knowing about because it fails *silently*:
`p_same_registered_pairs` arrives as **(1, n, n) per cell**, a leading singleton axis.
`pair_quality_rows` always stripped it, but the consistency check did not — so every
probabilistic run was declared inconsistent and the `if consistent` guard suppressed
**every** `PairQuality` row and left `min_p_same` NULL on every cell. The selector
`min_p_same > 0.9` would have returned nothing, with no error anywhere. It is now
`p_same_is_consistent()` with seven regression tests.

In [ ]:
# Completeness per run: stored vs implied, and the NaN gap.
(cr.CellRegResult * cr.CellRegRun).proj(
    'n_cells', 'n_pairs_expected', 'n_pairs_nonfinite', 'result_file_mtime',
    stored='n_pairs_expected - n_pairs_nonfinite') & 'n_pairs_nonfinite > 0'

---
## 4. Stage 2 — register the run  `# REQUIRES AUTHORISATION`

A run is identified by **the content of its result file**, not its path: 100 distinct
sha256 over 108 paths, because scan folders hold byte-identical backups. Keyed on path, one
registration would enter three times.

`CellRegRun.Session` carries `-> imaging.Curation`, so each row names the scan **and** the
suite2p paramset and curation it came from. A column that cannot be resolved to exactly one
curation goes to `CellRegRun.UnresolvedColumn` — it is never guessed.


In [ ]:
# Registration. RUN 1 EXECUTED THIS on 2026-10-20 for all 101 runs; re-running is
# idempotent (skip_duplicates) and will simply report the same rows.
#
# creg.activate(dj.config["custom"]["database.prefix"] + "cell_registration")
#
# sha = cellreg.file_sha256(CR)
# run_id = cellreg.run_id_from_sha(sha)
#
# creg.CellRegParamSet.register(
#     "leon_prob_sc_p03",
#     {"registration_approach": "probabilistic", "model_type": "spatial_correlation",
#      "p_same_threshold": meta["final_threshold"], "maximal_distance_um": meta["maximal_distance_um"],
#      "n_bins": meta["n_bins"], "initial_threshold": meta["initial_threshold"]},
#     label="Leon gratings/images standard",
#     matching_method="cellreg_probabilistic")
#
# creg.CellRegRun.insert1(dict(
#     cellreg_run_id=run_id, cellreg_paramset_id="leon_prob_sc_p03", subject="ROS-1881",
#     result_file=CR, result_file_sha256=sha, result_file_bytes=os.path.getsize(CR),
#     result_file_mtime=datetime.datetime.fromtimestamp(os.path.getmtime(CR)),
#     log_file=log, log_file_sha256=cellreg.file_sha256(log),
#     mat_format=cellreg.detect_mat_format(CR),
#     registration_approach=cellreg.normalise_registration_approach(meta["registration_approach"]),
#     alignment_mode=cellreg.normalise_alignment_mode(meta["alignment_type"]),
#     reference_session_index=meta.get("reference_session_index"),
#     n_sessions=meta["n_sessions"], n_registered_cells=meta["n_cells_final"],
#     ingested_at=datetime.datetime.now(), ingest_host=cellreg.ingest_host()))
#
# ... then one CellRegRun.Session row per resolved column, using
#     cellreg.resolve_column(column_masks, candidates, log_scan_id=...)
#     -> resolution_method, scan_key, mask_set_jaccard, ambiguous
print("not executed — see the banner at the top")

**Why Jaccard is computed even when the log resolves the column.** `Fall.mat` does **not**
update when suite2p curation does (Leon/Tobias, 2024-05-24), so a correctly-resolved column
can still carry a mask set that disagrees with the curation's. `mask_set_jaccard` is stored
unconditionally as a *staleness measurement*, not as a fallback.


---
## 5. Stage 3 — populate  `# REQUIRES AUTHORISATION`

`CellRegResult` is an `Imported` table: all `.mat` reading happens in its `make()`. It writes
per-cell quality as **rows with columns**, the cell↔session bridge, and `p_same` per cell per
session pair.

**Run it on a worker, not on tatchu3.** And watch the transaction: the largest run measures
~397,000 pair rows in a single `make()`.


In [ ]:
# Parsing. RUN 1 EXECUTED THIS; populate() is resumable, so re-running continues
# from wherever it stopped and never re-parses a finished run.
#
# creg.CellRegResult.populate(reserve_jobs=True, limit=50, max_calls=1, order="random",
#                             display_progress=True)
#
# A failed or inconsistent file is RECORDED, not skipped:
#   parse_ok = 0 with parse_error set, and no child rows.
#   p_same_consistent = 0 when the p_same matrix side != n_sessions, and no PairQuality rows.
print("not executed — see the banner at the top")

---
## 6. Stage 4 — querying

One edge attaches this schema to the pipeline: `CellRegRun.Session -> imaging.Curation`,
whose key is `(session_id, scan_id, paramset_idx, curation_id)`. `CellRegResult.Member`
inherits it and adds `mask`, so it lines up attribute-for-attribute with
`Segmentation.Mask` — joins need no renaming, and the curation can never be mixed implicitly.


In [ ]:
# Read-only patterns (will return nothing until the schema exists and is populated).
#
# run  = creg.CellRegRun & {"cellreg_run_id": run_id}
# cell = creg.CellRegResult.Cell & run & "cell_index = 42"
#
# # every (scan, mask) this cell is, across the sessions it was matched in
# members = creg.CellRegResult.Member & cell
#
# # the suite2p masks and traces those rows point at
# masks  = imaging.Segmentation.Mask   & members.proj()
# traces = imaging.Fluorescence.Trace  & members.proj() & "fluo_channel = 0"
#
# # cells tracked in >= 5 sessions whose weakest pairwise match is still good
# good = creg.CellRegResult.Cell & run & "n_sessions_present >= 5" & "min_p_same > 0.8"
#
# NOTE: n_sessions_present is a COUNT. Which sessions is a property of Member, not of Cell --
# Cell deliberately carries no session-scoped attribute. Partial tracking is the normal case:
# median sessions-per-cell is 1 in most runs.
print("query patterns above are illustrative")

---
## 7. Worked example — one site, four cells, eleven sessions (read-only)

Site **`sess9FPXNI8U`** (ROS-1841 → ROS-1881 lineage; a `t78` validation row). The site holds
**26 sessions**, of which **11 are gratings scans** with 12 directions each, recorded every two
days from 2024-09-05 to 2024-09-25. All of them are `paramset_idx 10, curation_id 2` —
**curation 1 carries no `roi_group` at all**, which is the first thing to check on any site.

Of the cells in those 11 scans, **305 are present in all 11**. That is the set cell
registration exists to give you, and the figure below follows four of them across the series.

Everything here reads tables that exist **today** — `MotionCorrection.Summary` for the
template, `Segmentation.Mask` for outlines and `roi_group`, `Activity.Trace` for responses.
No `cell_registration` table is needed to *read* longitudinal identity; what the schema adds is
knowing **which run produced these groupings, with which parameters, and how well each cell
matched**, none of which `roi_group` can tell you.


In [ ]:
SID, PIDX, CID = "sess9FPXNI8U", 10, 2

# The site's gratings scans, chronologically. curation_id 2 is the one carrying roi_group
# here -- curation 1 has none at all, which is the first thing to check on any site.
rows = dj.conn().query("""
    SELECT DISTINCT m.session_id, m.scan_id, s2.session_datetime
    FROM roselab_imaging.__segmentation__mask m
    JOIN roselab_session.session_same_site s ON s.session_id  = m.session_id
    JOIN roselab_session.session          s2 ON s2.session_id = m.session_id
    WHERE s.same_site_id = %s AND m.roi_group > 0
      AND m.paramset_idx = %s AND m.curation_id = %s
    ORDER BY s2.session_datetime""", (SID, PIDX, CID)).fetchall()

sessions = []
for sess, scan, dt in rows:
    ets = sorted(e for e in set((event.Event & {"session_id": sess, "scan_id": scan}
                                 ).fetch("event_type")) if e.startswith("Moving_Gratings:"))
    if ets:                                   # gratings only: the site also holds image scans
        sessions.append((sess, scan, ets, dt))
dates = [str(dt)[:10] for *_, dt in sessions]

print(f"{len(sessions)} gratings scans in site {SID}:")
for (sess, scan, ets, _), d in zip(sessions, dates):
    print(f"   {d}  {sess}  {scan}  ({len(ets)} directions)")

def roi_to_mask(sess, scan):
    """roi_group -> mask for one scan. The mask NUMBER differs between sessions;
    roi_group is what ties them together."""
    return {int(g): int(m) for g, m in dj.conn().query(
        """SELECT roi_group, mask FROM roselab_imaging.__segmentation__mask
           WHERE session_id=%s AND scan_id=%s AND paramset_idx=%s AND curation_id=%s
             AND roi_group > 0""", (sess, scan, PIDX, CID)).fetchall()}

per_scan = [set(roi_to_mask(s, sc)) for s, sc, _, _ in sessions]
common = sorted(set.intersection(*per_scan))
print(f"\ncells present in all {len(sessions)} scans: {len(common)}")

### The trap that costs an afternoon

**Cascade traces are NaN-padded at both ends** (wiki issue **T9**). A plain `.mean()` over a
trial window propagates the NaN, every response becomes NaN, every preferred orientation
becomes 0°, and the whole arm silently reports *no tuning* — for a non-biological reason.

Use NaN-aware means and require each window to be at least half finite. This notebook's
figure was empty on the first attempt for exactly this reason.


In [ ]:
from scipy import stats

CELLS = [231, 3, 38, 34]          # four of the `common` cells, ranked in the first session only

def scan_context(sess, scan, ets):
    """Frame clock, per-direction trial onsets, and the best signal available for one scan."""
    frame_t = np.sort((event.Event & {"session_id": sess, "scan_id": scan,
                        "event_type": "bench2p_frames"}).fetch("event_start_time").astype(float))
    fps = 1.0 / np.median(np.diff(frame_t))
    onset = {}
    for e in ets:
        d = int(re.match(r"Moving_Gratings:(\d+);", e).group(1))
        t0 = (event.Event & {"session_id": sess, "scan_id": scan,
                             "event_type": e}).fetch("event_start_time")
        onset.setdefault(d, []).extend(int(np.searchsorted(frame_t, float(t))) for t in t0)
    avail = set((imaging.Activity & {"session_id": sess, "scan_id": scan,
                 "paramset_idx": PIDX, "curation_id": CID}).fetch("extraction_method"))
    method = next((m for m in ("cascade_inference_25ms_filter", "cascade_inference_11",
                               "suite2p_deconvolution") if m in avail), None)
    return fps, onset, method

def windows(trace, onsets, PRE, POST, STIM, nF):
    """Trial windows around onset, keeping only those at least half finite (wiki issue T9)."""
    rows = [trace[f - PRE:f + POST] for f in onsets if f - PRE >= 0 and f + POST <= nF]
    return [r for r in rows if np.isfinite(r[:PRE]).mean() >= 0.5
            and np.isfinite(r[PRE:PRE + STIM]).mean() >= 0.5]

# window geometry, taken from the first session (all eleven run at ~29.98 Hz)
fps_ref, onset_ref, method_ref = scan_context(*sessions[0][:3])
DIRS = sorted(onset_ref)
PRE, POST, STIM = (int(round(1.0 * fps_ref)), int(round(4.0 * fps_ref)), int(round(3.0 * fps_ref)))
print(f"signal {method_ref} | fps {fps_ref:.2f} | directions {DIRS}")
print(f"window: {PRE} frames baseline + {POST} frames after onset, {STIM}-frame grating")

In [ ]:
# ---- per-session direction responses for the four cells ---------------------
panels = {}                                  # (roi_group, session index) -> {direction: (mean, ci)}
for si, (sess, scan, ets, _) in enumerate(sessions):
    fps_s, onset_s, method_s = scan_context(sess, scan, ets)
    P, O, S = int(round(1.0*fps_s)), int(round(4.0*fps_s)), int(round(3.0*fps_s))
    g2m = roi_to_mask(sess, scan)
    tr = (imaging.Activity.Trace
          & {"session_id": sess, "scan_id": scan, "paramset_idx": PIDX,
             "curation_id": CID, "extraction_method": method_s}
          & [{"mask": g2m[g]} for g in CELLS if g in g2m]).fetch(
          "mask", "activity_trace", as_dict=True)
    tt = {int(r["mask"]): np.asarray(r["activity_trace"], np.float32) for r in tr}
    nF = len(next(iter(tt.values())))
    for g in CELLS:
        t_ = tt.get(g2m.get(g, -1))
        if t_ is None:
            continue
        per_dir = {}
        for d in DIRS:
            w = windows(t_, onset_s[d], P, O, S, nF)
            if len(w) < 10:
                continue
            R = np.vstack(w).astype(np.float64)
            R = np.where(np.isfinite(R), R, np.nanmean(R))       # T9: never a bare mean
            n = R.shape[0]
            per_dir[d] = (R.mean(0),
                          stats.t.ppf(0.975, n - 1) * R.std(0, ddof=1) / np.sqrt(n))
        panels[(g, si)] = per_dir
    print(f"   [{si+1}/{len(sessions)}] {sess} {method_s} fps {fps_s:.2f} "
          f"-> {sum(1 for g in CELLS if panels.get((g, si)))}/{len(CELLS)} cells")

print("\npanels:", len(panels), "= cells x sessions")

In [ ]:
# ---- per-session cutouts: the same cell, re-segmented in each session --------
CROP = 16                                          # 32x32 px, ~38 um: tight on the soma
cutout  = np.full((len(CELLS), len(sessions), 2*CROP, 2*CROP), np.nan, np.float32)
outline = np.zeros_like(cutout, bool)

for si, (sess, scan, _, _) in enumerate(sessions):
    k = {"session_id": sess, "scan_id": scan, "paramset_idx": PIDX, "curation_id": CID}
    avg = np.asarray((imaging.MotionCorrection.Summary & k).fetch1("average_image"), np.float32)
    g2m = roi_to_mask(sess, scan)
    rows = (imaging.Segmentation.Mask & k
            & [{"mask": g2m[g]} for g in CELLS if g in g2m]).fetch(
            "roi_group", "mask_xpix", "mask_ypix", as_dict=True)
    byg = {int(r["roi_group"]): r for r in rows}
    for ci_, g in enumerate(CELLS):
        r = byg.get(g)
        if r is None:
            continue                               # cell absent from this session
        xp = np.asarray(r["mask_xpix"]).ravel().astype(int)
        yp = np.asarray(r["mask_ypix"]).ravel().astype(int)
        # centre on the mask's own pixels, not mask_center_x/y -- they agree to a
        # pixel or two, but the pixels are what gets drawn
        x0 = int(np.clip(round(xp.mean()) - CROP, 0, avg.shape[1] - 2*CROP))
        y0 = int(np.clip(round(yp.mean()) - CROP, 0, avg.shape[0] - 2*CROP))
        cutout[ci_, si] = avg[y0:y0+2*CROP, x0:x0+2*CROP]
        inx = (xp >= x0) & (xp < x0+2*CROP) & (yp >= y0) & (yp < y0+2*CROP)
        outline[ci_, si][yp[inx]-y0, xp[inx]-x0] = True

print("cutouts:", cutout.shape, "| cells found:", int(np.isfinite(cutout).any((2,3)).sum()),
      "of", cutout.shape[0]*cutout.shape[1])

# mask size varies session to session -- each session is segmented independently;
# registration links the ROIs, it does not make them identical.
for ci_, g in enumerate(CELLS):
    print(f"   cell {g}: npix per session {[int(outline[ci_, s].sum()) for s in range(len(sessions))]}")

In [ ]:
# ---- the figure: portrait, one cell per column, sessions down the page -------
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy.ndimage import gaussian_filter1d

GAP, SM = 14, 0.15 * fps_ref                  # gap between directions; display smoothing
FOCAL, BAND, STIMC = "#12355b", "#bdd0e3", "#e6edf5"
UMPX = 615.6 / 512                            # µm per pixel for this objective
NS, ND, NT, D = len(sessions), len(DIRS), PRE + POST, 2 * CROP

wr, cols = [], []
for i in range(len(CELLS)):
    cols.append((len(wr), len(wr) + 1)); wr += [0.46, 1.15]
    if i < len(CELLS) - 1:
        wr += [0.20]                          # spacer between cell blocks

fig = plt.figure(figsize=(6.9, 9.3))
gs = fig.add_gridspec(NS, len(wr), width_ratios=wr, hspace=0.10, wspace=0.0,
                      left=0.105, right=0.994, top=0.930, bottom=0.052)

for ci_, g in enumerate(CELLS):
    cimg, ctr = cols[ci_]
    peak = max(np.nanmax(m + h) for (gg, si) in panels if gg == g
               for m, h in panels[(gg, si)].values())      # one scale per cell
    for si in range(NS):
        axi = fig.add_subplot(gs[si, cimg])
        lo, hi = np.nanpercentile(cutout[ci_, si], [2, 99.5])
        axi.imshow(cutout[ci_, si], cmap="gray", vmin=lo, vmax=hi, interpolation="bilinear")
        axi.contour(outline[ci_, si], levels=[0.5], colors=FOCAL, linewidths=0.9)
        axi.set_axis_off()
        if ci_ == 0:
            axi.text(-0.14, 0.5, dates[si][5:].replace("-", "/"), transform=axi.transAxes,
                     ha="right", va="center", fontsize=6.2, color="0.25")
        if si == NS - 1 and ci_ == 0:
            px = 10 / UMPX
            axi.plot([2.5, 2.5 + px], [D - 2.5] * 2, lw=1.6, color="w", solid_capstyle="butt")
            axi.text(2.5 + px / 2, D - 3.8, "10 µm", ha="center", va="bottom", fontsize=5.2,
                     color="w", path_effects=[pe.withStroke(linewidth=1.0, foreground="0.25")])

        axt = fig.add_subplot(gs[si, ctr])
        per_dir = panels.get((g, si), {})
        for di, d in enumerate(DIRS):
            if d not in per_dir:
                continue
            m_, h_ = (gaussian_filter1d(a.astype(float), SM, mode="nearest") for a in per_dir[d])
            xs = np.arange(m_.size) + di * (NT + GAP)
            axt.add_patch(mpl.patches.Rectangle((di * (NT + GAP) + PRE, -0.08), STIM, 1.16,
                                                facecolor=STIMC, edgecolor="none", zorder=0))
            axt.fill_between(xs, (m_ - h_) / peak, (m_ + h_) / peak, color=BAND, lw=0, zorder=1)
            axt.plot(xs, m_ / peak, color=FOCAL, lw=0.6, zorder=2)
        axt.set_xlim(-GAP, ND * (NT + GAP) - GAP + 2)
        axt.set_ylim(-0.13, 1.07)
        axt.set_axis_off()
        if si == NS - 1 and ci_ == 0:
            axt.plot([PRE, PRE + fps_ref], [-0.105] * 2, lw=1.5, color="0.15",
                     solid_capstyle="butt")
            axt.text(PRE + fps_ref / 2, -0.15, "1 s", ha="center", va="top",
                     fontsize=5.6, color="0.15")

    b0 = gs[0, cimg].get_position(fig); b1 = gs[0, ctr].get_position(fig)
    fig.text((b0.x0 + b1.x1) / 2, 0.940, f"cell {int(g)}", ha="center", va="bottom",
             fontsize=7.5, color="0.08")

fig.text(0.012, 0.5, "session", rotation=90, ha="left", va="center", fontsize=7, color="0.30")
fig.text(0.5, 0.992, f"Four cells tracked across {NS} sessions of site {SID} "
         f"(ROS-1881), {dates[0]} to {dates[-1][5:]}", ha="center", va="top",
         fontsize=8, color="0.08")
fig.text(0.5, 0.030, "per panel: 12 directions 0–330° left to right, 1 s baseline then a 3 s "
         "drifting grating (shaded), mean ± 95% CI over 15 trials",
         ha="center", va="bottom", fontsize=5.8, color="0.36")
fig.text(0.5, 0.006, "Cascade inferred rate (25 ms filter), 150 ms smoothing  ·  each cell "
         "scaled once by its peak across all sessions  ·  cell = roi_group, "
         f"paramset {PIDX}, curation {CID}", ha="center", va="bottom", fontsize=5.8, color="0.36")

import os; os.makedirs("figures", exist_ok=True)
fig.savefig("figures/fig_cellreg_longitudinal.png", dpi=400)   # gitignored; regenerate by running
print("panels:", len(fig.axes))

### Result

![Four cells tracked across 11 sessions]({{artifact:72cfa7a6-6e04-4eb7-8c83-67d00d7fe04d}})

One column per cell, one row per session, oldest at the top. Each row pairs the cell's own
cutout from **that session's** template — re-segmented independently, outline and all — with its
12-direction sweep from that session.

Two things are worth reading off it:

**The cutouts are the registration, made visible.** The same soma appears in the same place in
all eleven, which is what `roi_group` is asserting. The outlines are *not* identical between
sessions — mask size varies by a factor of two for some cells (printed above) — because each
session is segmented on its own. Registration links ROIs; it does not make them the same ROI.

**The responses are scaled per cell, once, across all eleven sessions** — never per panel. Cell
231 keeps a clear preferred direction throughout. Cell 34 is weak across the whole series at its
own scale. Per-panel normalisation would have made every row look alike and erased exactly the
comparison the figure exists to support.

What this figure cannot tell you is whether a weak session means the cell went quiet, drifted
out of plane, or was mis-matched by CellReg. That is the gap `cell_registration` fills:
`CellRegResult.Cell.min_p_same` and `CellRegResult.PairQuality` carry the match evidence per
cell and per session pair, so a responsiveness change can be separated from a registration
artefact instead of being confounded with one.

*(`notebooks/figures/*.png` is gitignored, so the figure is not committed — running the cells
above regenerates it.)*


---
## 8. Gotchas, in the order they will bite you

| | |
|---|---|
| **Cascade NaN padding (T9)** | NaN-pad at both ends; a plain mean silently yields zero tuning. Use NaN-aware means, require ≥50 % finite. |
| **`suite2p_deconvolution` is sparse** | Fine for statistics, unreadable as a plotted trace. Use a Cascade variant for figures. |
| **Table-name prefixes** | `MotionCorrection` is `Imported` → `_motion_correction__summary`. `Segmentation` is `Computed` → `__segmentation__mask`. |
| **Blobs need DataJoint** | Fetch through virtual modules or the activated pipeline. Raw SQL + `datajoint.blob.unpack` fails on externally-stored blobs. |
| **`event.Event` is ~1.05 billion rows** | Always restrict by `session_id` *and* `scan_id` first (wiki **D5**). An unrestricted count will hang. |
| **Natural-image vs gratings scans** | Same site, different scans, **separate CellReg runs**. `scan9FPZHKKB` is images; `scan9FPZGABI` is gratings. |
| **Two runs in one `Results/`** | ROS-1985 has a 7-session run and a 14-session run side by side (wiki **D3**). Pair the log to the result file; do not assume one run per folder. |
| **`same_site_id` cannot say "unknown"** | Populated for all 3,460 sessions but 74 % self-reference. 41 sites carry real groupings (up to 66 sessions). Treat a self-reference as unknown, not as a singleton site. |
| **Not on tatchu3** | CellReg and bulk `.mat` parsing go to Loki or ibehaveGPU. |


---
## 9. What is deliberately *not* here

- **Eligibility thresholds.** Registration stores continuous match quality only; deciding
  which cells are good enough is a QC/cohort act with its own version history.
- **`MaskCoordinates.make()`** raises `NotImplementedError` pending verification that
  `centroid_locations_corrected` is ordered consistently with `Segmentation.Mask` — declaring
  it without that check would invite wiki issue **L8**.
- **Writing `roi_group`.** `RoiGroupProjection` records what a projection *would* be;
  performing one is a production write needing explicit authorisation.
